# 04 · LightGBM

Es un modelo de gradient boosting el cuyal  agrega árboles pequeños en secuencia, cada uno corrigiendo el error de los anteriores.

Se entrena con junio, se evalúa con septiembre y solo en las localidades habilitadas. Se comparan tres conjuntos de variables (`variables_v2.CONJUNTOS`): `v1` (las del cuaderno 02), `v2` (v1 + tipología, más amenidades, noches mínimas recortadas y estancia mensual) y `v2_geo` (v2 + latitud y longitud).

Además del MdAPE sobre toda la captura se registra `mdape_nuevos`, que es el error en los anuncios que no existían en junio (una simulación del caso real, dado que son anuncios nuevos). Las corridas van al experimento `bogoprice-lightgbm` de MLflow.

In [ ]:
import lightgbm as lgb
import pandas as pd

from bogoprice.experimentos import evaluar, preparar_datos
from bogoprice.tracking import iniciar_mlflow

# Tag de autor de MLflow
AUTOR = "Edgar"
iniciar_mlflow("bogoprice-lightgbm")
datos = preparar_datos()
resultados = []


def lgbm(**p):
    # Valores base: cada prueba cambia solo uno de ellos
    base = {"objective": "regression", "n_estimators": 1500, "learning_rate": 0.03,
            "num_leaves": 63, "max_depth": -1, "min_child_samples": 20,
            "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 0.7,
            "reg_lambda": 0.0, "reg_alpha": 0.0}
    return lgb.LGBMRegressor(verbose=-1, random_state=0, **(base | p)), base | p

## 1. Conjuntos de variables

La configuración base con los tres conjuntos de variables.

In [ ]:
for version in ["v1", "v2", "v2_geo"]:
    modelo, params = lgbm()
    m = evaluar(datos, f"lgbm_{version}", modelo, version, {"modelo": "LightGBM", **params}, AUTOR)[0]
    resultados.append({"corrida": f"lgbm_{version}", "conjunto": version,
                       "parametro": "base", "valor": "-", **m})
    print(f"{version:7s} MdAPE nuevos = {m['mdape_nuevos']:.2f} %")

## 2. Un hiperparámetro a la vez (v2 con y sin coordenadas)

Se parte de la configuración base y se prueban **4 valores de cada hiperparámetro**, dejando los demás constantes. Así cada corrida muestra el efecto de un solo parámetro.

| Hiperparámetro | Qué controla | Base | Valores probados |
|---|---|---|---|
| `objective` | Función de pérdida que se minimiza | `regression` (cuadrática) | `l1`, `huber`, `fair`, `poisson` |
| `n_estimators` | Número de árboles | 1500 | 300, 800, 2500, 4000 |
| `learning_rate` | Cuánto corrige cada árbol | 0,03 | 0,01, 0,02, 0,05, 0,1 |
| `num_leaves` | Hojas por árbol (complejidad) | 63 | 15, 31, 127, 255 |
| `max_depth` | Profundidad máxima (-1 = sin límite) | -1 | 4, 6, 8, 12 |
| `min_child_samples` | Anuncios mínimos por hoja | 20 | 5, 10, 50, 100 |
| `subsample` | Fracción de anuncios por árbol | 0,8 | 0,5, 0,6, 0,7, 1,0 |
| `colsample_bytree` | Fracción de variables por árbol | 0,7 | 0,3, 0,5, 0,9, 1,0 |
| `reg_lambda` | Regularización L2 | 0 | 0,1, 1, 10, 50 |
| `reg_alpha` | Regularización L1 | 0 | 0,1, 1, 5, 10 |

In [ ]:
# Cada parámetro se prueba con 4 valores; los demás quedan en la base
pruebas = {
    "objective": ["l1", "huber", "fair", "poisson"],
    "n_estimators": [300, 800, 2500, 4000],
    "learning_rate": [0.01, 0.02, 0.05, 0.1],
    "num_leaves": [15, 31, 127, 255],
    "max_depth": [4, 6, 8, 12],
    "min_child_samples": [5, 10, 50, 100],
    "subsample": [0.5, 0.6, 0.7, 1.0],
    "colsample_bytree": [0.3, 0.5, 0.9, 1.0],
    "reg_lambda": [0.1, 1, 10, 50],
    "reg_alpha": [0.1, 1, 5, 10],
}

for version in ["v2", "v2_geo"]:
    for parametro, valores in pruebas.items():
        for valor in valores:
            modelo, params = lgbm(**{parametro: valor})
            nombre = f"lgbm_{parametro}{valor}_{version}"
            m = evaluar(datos, nombre, modelo, version,
                        {"modelo": "LightGBM", "parametro_variado": parametro, **params}, AUTOR)[0]
            resultados.append({"corrida": nombre, "conjunto": version, "parametro": parametro,
                               "valor": valor, **m})
        print(f"{version:7s} {parametro} listo")

In [ ]:
tabla = pd.DataFrame(resultados)
tabla.pivot_table(index=["parametro", "valor"], columns="conjunto", values="mdape_nuevos").round(2)